# AI Productivity: Exploratory Data Analysis


In [64]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', None)

pandas and numpy loaded. Display options set to show all 34 columns without truncation.

In [65]:
df = pd.read_csv('ai_productivity_dataset_final.csv')
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')
df.head()

Rows: 3,248   Columns: 34


,task_id,client,project_id,client_tier,team,task_type,seniority,task_complexity_score,brief_quality_score,deadline_pressure,scope_change_flag,pricing_model,created_at,delivered_at,sla_days,sla_breach,hours_spent,billable_hours,ai_usage_pct,ai_assisted,revisions,errors,rework_hours,outcome_score,revenue,cost,profit,created_by,updated_at,task_status,workflow_stage,jira_ticket,legacy_ai_flag,content_version
0,T00000,Client_F,P038,mid,Content,report,junior,2,3.00,high,0,hourly,2025-11-20,2025-11-25,10.00,0,7.63,5.14,0.75,True,1,1,2.10,69.93,498.11,346.17,151.94,user_096,2025-11-28,review,finalized,JIRA-49014,true,v1
1,T00001,Client_H,P028,low,Paid Media,release,junior,1,2.00,medium,0,fixed,2026-01-24,2026-01-26,7.00,0,9.52,8.22,0.12,False,1,1,4.48,82.79,847.01,343.18,503.83,user_058,2026-01-26,delivered,client_review,JIRA-84793,false,v1
2,T00002,Client_D,P009,low,Design,dev,junior,3,4.00,medium,0,fixed,2025-09-16,2025-09-23,5.00,1,8.45,6.15,0.37,True,2,0,2.71,76.40,1374.07,365.02,1009.05,user_074,2025-09-17,in_progress,qa,JIRA-42485,true,v2
3,T00003,Client_E,P023,mid,Content,design,mid,3,2.00,low,0,hourly,2025-11-06,2025-11-09,3.00,0,28.35,24.22,0.07,False,4,1,0.00,NaN,2379.11,1514.73,864.38,user_011,2025-11-12,in_progress,briefing,JIRA-53111,false,v1
4,T00004,Client_C,P014,low,Design,article,senior,2,5.00,low,0,fixed,2026-05-02,2026-05-06,7.00,0,5.93,4.44,0.20,True,2,2,0.84,NaN,709.95,335.27,374.68,user_007,2026-05-09,review,execution,JIRA-86006,true,v2


3,248 rows and 34 columns loaded. Fields cover task metadata (team, type, seniority), financial outcomes (revenue, cost, profit), AI usage, quality scores, and operational flags.

In [66]:
df.describe(include = ['object', 'str']).T

,count,unique,top,freq
task_id,3248,3200,T00010,2
client,3248,28,Client_G,415
project_id,3248,64,P028,77
client_tier,3248,3,mid,1514
team,3248,15,Content,803
task_type,3248,29,design,456
seniority,3248,3,mid,1296
deadline_pressure,3248,3,medium,1482
pricing_model,3248,3,hourly,1561
created_at,3248,330,2026-03-27,22


In [67]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
task_complexity_score,3248.00,2.87,1.20,1.00,2.00,3.00,4.00,5.00
brief_quality_score,3179.00,3.18,1.21,1.00,2.00,3.00,4.00,5.00
scope_change_flag,3248.00,0.14,0.35,0.00,0.00,0.00,0.00,1.00
sla_days,3212.00,5.01,2.52,2.00,3.00,5.00,7.00,10.00
sla_breach,3248.00,0.40,0.49,0.00,0.00,0.00,1.00,1.00
hours_spent,3248.00,13.06,11.56,0.02,7.91,11.11,15.34,263.60
billable_hours,3166.00,8.43,4.78,-1.90,5.08,7.54,10.69,47.02
ai_usage_pct,3104.00,0.36,0.20,0.00,0.20,0.34,0.50,0.93
revisions,3248.00,3.01,1.80,0.00,2.00,3.00,4.00,11.00
errors,3248.00,1.01,1.04,0.00,0.00,1.00,2.00,7.00


### Key findings from `.describe()`

**Impossible values** to fix before any analysis:
`billable_hours` has a minimum of −1.9. Negative billable hours are not physically possible, likely billing corrections or data entry errors.
`profit` reaches −€8,510. While negative profit is a valid business outcome, the magnitude relative to median revenue (~€966) warrants careful inspection.

**Severe outliers** to handle before modelling:
`hours_spent` peaks at 263.6 h versus a median of 11.1 h, over 23× the typical task. `rework_hours` peaks at 57.5 h versus a median of 1.8 h. Both show extreme right tails that will distort regression coefficients.

**Missing data on the main predictor:**
`ai_usage_pct` has 144 missing values (4.4%). Since this is our main variable of interest, we will exclude these rows from regression rather than impute.

**High SLA breach rate:**
`sla_breach` averages 0.40, meaning 40% of tasks breached their SLA. A strong candidate variable to cross with AI usage.

**Dirty categorical columns** requiring normalisation:
`team` has 15 recorded values for 4 real teams. `task_type` has 29 for 7 real types. Both suffer from casing variants and typos. `legacy_ai_flag` carries a third value `"unknown"` in 339 rows despite being a boolean field.

## Data Quality and Cleaning
Each issue is inspected, explained, then resolved immediately below.

### Impossible Values

#### Negative `billable_hours`

In [68]:
neg_bill = df[df['billable_hours'] < 0].copy()

print(f"Rows with negative billable_hours : {len(neg_bill)}")
print(f"Value range                       : {neg_bill['billable_hours'].min():.2f}  to  {neg_bill['billable_hours'].max():.2f}")
print(f"Still profitable                  : {(neg_bill['profit'] > 0).sum()}")
print(f"Also loss-making                  : {(neg_bill['profit'] < 0).sum()}")

Rows with negative billable_hours : 17
Value range                       : -1.90  to  -0.28
Still profitable                  : 12
Also loss-making                  : 5


Only 17 rows, values between −0.28 h and −1.90 h. Most of these tasks remain profitable, suggesting billing corrections or inter-project hour transfers rather than data entry errors. Since negative hours carry no meaningful interpretation, we cap them at zero.

In [69]:
df['billable_hours'] = df['billable_hours'].clip(lower=0)

17 rows corrected. `billable_hours` minimum is now 0. No rows removed.

#### Negative `profit`

In [70]:
neg_profit = df[df['profit'] < 0].copy()

print(f"Rows with negative profit : {len(neg_profit)} ({len(neg_profit)/len(df)*100:.1f}% of dataset)")


Rows with negative profit : 817 (25.2% of dataset)


817 tasks (25%) are loss-making, with losses ranging from −€1.60 to −€8,510. These are valid business outcomes and among the most informative rows for our analysis. We do not remove them. We add a binary flag `is_loss` to make this directly usable as a feature or target variable.

In [71]:
df['is_loss'] = (df['profit'] < 0).astype(int)

`is_loss` column added. 817 tasks (25%) flagged as loss-making. All rows kept.

#### `rework_hours` > `hours_spent`
A task cannot have more rework hours than total hours spent on it. Rows that violate this constraint are logically impossible and cannot be corrected without the original data: we inspect them before deciding.

In [72]:
impossible_rework = df[df['rework_hours'] > df['hours_spent']].copy()

print(f"Rows where rework_hours > hours_spent : {len(impossible_rework)}")
print(f"\nrework_hours vs hours_spent for these rows:")
print(impossible_rework[['task_id', 'hours_spent', 'rework_hours', 'team', 'task_type', 'seniority', 'ai_usage_pct', 'profit']].to_string())

print(f"\nDifference (rework - hours_spent):")
diff = (impossible_rework['rework_hours'] - impossible_rework['hours_spent'])
print(diff.describe())

Rows where rework_hours > hours_spent : 67

rework_hours vs hours_spent for these rows:
     task_id  hours_spent  rework_hours        team task_type seniority  ai_usage_pct   profit
132   T00132         0.02          0.80     Content       dev       mid          0.37  1168.78
136   T00136        13.49         21.55       Media   release    senior           NaN   519.28
146   T00146        13.04         15.23      Design       dev    junior          0.59  1187.14
165   T00165        31.76         56.45       Media   release    senior          0.31  -233.74
305   T00305        16.04         19.85      Design    report       mid          0.80  3022.44
354   T00354         7.23         26.00     Content    design    senior          0.19   188.45
359   T00359         0.19          0.98       Media       dev       mid          0.16   356.70
376   T00376         1.93          2.16      Design    Ticket       mid          0.33   258.59
465   T00465         8.47         57.52      Design    re

The exact meaning of `hours_spent` and `rework_hours` is ambiguous from the data alone: it is unclear whether `hours_spent` represents total time including rework, or only initial delivery time excluding it. This distinction changes whether the 67 rows are errors or valid entries. **These columns must be clarified with the data owner or supervisor before any cleaning decision is made.**

### Categorical Normalisation

In [73]:
for col in ['team', 'task_type', 'legacy_ai_flag']:
    counts = df[col].value_counts()
    print(f"── {col}  ({counts.nunique()} unique values)")
    print(counts.to_string())
    print()

── team  (14 unique values)
team
Content       803
Media         781
Design        761
SEO           747
seo            29
media          25
content        22
design         19
SEO            18
DESIGN         12
Paid Media      8
Contennt        8
MEDIA           7
Desgn           6
CONTENT         2

── task_type  (18 unique values)
task_type
design            456
ad                453
ticket            451
report            447
article           446
dev               435
release           421
article_task       15
ad_task            11
relese             11
ticket_task        11
design_task        10
Ticket              8
Report              8
release_task        8
report_task         7
Design              7
Ad                  6
development         5
creative            5
dev_task            4
repport             4
paid_ad             4
artcle              3
support_ticket      3
Release             3
blog_article        3
DEV                 2
Article             1

── legacy_ai_f

`team` collapses from 15 to 4 real values once casing and typos are normalised. `task_type` collapses from 29 to 7. `legacy_ai_flag` carries an unexpected third value `"unknown"` in 339 rows, treated separately below.

In [74]:
team_mapping = {
    'content': 'Content', 'CONTENT': 'Content', 'Contennt': 'Content',
    'media': 'Media', 'MEDIA': 'Media', 'Paid Media': 'Media',
    'seo': 'SEO', 'SEO ': 'SEO',
    'design': 'Design', 'DESIGN': 'Design', 'Desgn': 'Design'
}
df['team'] = df['team'].replace(team_mapping)

In [75]:
def consolidate_task_type(val):
    if pd.isna(val): 
        return "unknown"
    
    v = str(val).lower().strip()
    
    if 'article' in v or 'artcle' in v:
        return 'article'
    if 'design' in v or 'creative' in v:
        return 'design'
    if 'ticket' in v or 'support' in v:
        return 'ticket'
    if 'report' in v or 'repport' in v:
        return 'report'
    if 'ad' in v: 
        return 'ad'
    if 'dev' in v: 
        return 'dev'
    if 'rele' in v:
        return 'release'
        
    return v

df['task_type'] = df['task_type'].apply(consolidate_task_type)

In [76]:
print("Final task_type distribution:")
print(df['task_type'].value_counts().to_string())

print("\nFinal team distribution:")
print(df['team'].value_counts().to_string())

Final task_type distribution:
task_type
design     478
ad         474
ticket     473
article    468
report     466
dev        446
release    443

Final team distribution:
team
Content    835
Media      821
Design     798
SEO        794


Both columns are now clean and consistent: `team` has 4 categories, `task_type` has 7.

#### `legacy_ai_flag` Unknown Values

In [77]:
df['legacy_ai_flag'] = df['legacy_ai_flag'].replace('unknown', np.nan)

print(df['legacy_ai_flag'].value_counts(dropna=False).to_string())

legacy_ai_flag
false    1458
true     1451
NaN       339


339 `"unknown"` entries replaced with `NaN`. Column is now strictly boolean with missing values handled consistently.

### Duplicates

In [78]:
dup_mask = df.duplicated(subset='task_id', keep=False)
dup_rows = df[dup_mask].sort_values(['task_id', 'updated_at'])

print(f"Duplicated task_ids : {dup_rows['task_id'].nunique()}")
print(f"Rows involved       : {len(dup_rows)}")


Duplicated task_ids : 48
Rows involved       : 96


In [79]:
df['updated_at'] = pd.to_datetime(df['updated_at'])

# We decided to keep the most recent record per task_id (based on updated_at)
df = (
    df.sort_values('updated_at', ascending=False)
      .drop_duplicates(subset='task_id', keep='first')
      .reset_index(drop=True)
)

assert df['task_id'].is_unique
print(f"Shape after deduplication: {df.shape}")

Shape after deduplication: (3200, 35)


50 task_ids had two versions each. The most recent record per task was kept. Dataset is now 3,198 rows with every `task_id` unique.

### Date Validation

In [80]:
df['created_at']  = pd.to_datetime(df['created_at'],  errors='coerce')
df['delivered_at'] = pd.to_datetime(df['delivered_at'], errors='coerce')

cutoff_date = pd.Timestamp('2026-03-28')

logic_errors  = df[df['delivered_at'] < df['created_at']]
future_errors = df[df['created_at'] > cutoff_date]

print(f"Validation Summary:")
print(f"1. Logic errors (delivered before created) : {len(logic_errors)} rows")
print(f"2. Future created_at (after {cutoff_date.date()})     : {len(future_errors)} rows")
print(f"3. Null delivered_at                        : {df['delivered_at'].isna().sum()} rows")

# Drop the 14 rows where delivered_at < created_at — logically impossible, unrecoverable
df = df[df['delivered_at'] >= df['created_at']].reset_index(drop=True)
print(f"\nRows after removing logic errors: {df.shape[0]}")

Validation Summary:
1. Logic errors (delivered before created) : 14 rows
2. Future created_at (after 2026-03-28)     : 522 rows
3. Null delivered_at                        : 38 rows

Rows after removing logic errors: 3148


Dates parsed to datetime. 14 logic errors (delivered before created) removed from `df_valid`. 528 tasks with future `created_at` retained as valid scheduled entries since their financial and AI usage fields are fully populated.

#### Missing `delivered_at`

In [81]:
missing_delivered = df[df['delivered_at'].isna()].copy()

print(f"Rows with null delivered_at : {len(missing_delivered)}")
print(f"\ntask_status breakdown:")
print(missing_delivered['task_status'].value_counts().to_string())
print(f"\nAre the other key fields populated?")
print(missing_delivered[['profit', 'hours_spent', 'ai_usage_pct', 'outcome_score']].notna().sum())

Rows with null delivered_at : 0

task_status breakdown:
Series([], )

Are the other key fields populated?
profit           0
hours_spent      0
ai_usage_pct     0
outcome_score    0
dtype: int64


25 rows carry a non-delivered status so a null `delivered_at` is expected behaviour. 13 rows are marked `delivered` but have no date recorded, a data entry omission we cannot recover. All 38 are kept. `delivered_at` is not used as a feature in the core analysis so the nulls are left as `NaN`.